In [ ]:
from __future__ import annotations

import jax
import jax.numpy as jnp
import equinox as eqx
import numpy as np
import optax
import pandas as pd
from scipy.integrate import solve_ivp
import matplotlib.pyplot as plt

from mxlpy import Model, Simulator
from mxlpy.jax import UDE, fit as fit_jax

# Universal Differential Equations for Plant Fluorescence

This notebook demonstrates a complete **UDE** (Universal Differential Equation) workflow
using mxlpy's JAX export.

**Scenario:**  A small mechanistic model captures the quenching dynamics of plant
chlorophyll fluorescence under a changing illumination protocol.  The *true* biological
system contains an unknown periodic forcing — simulating e.g. stomatal oscillations or
circadian rhythm crosstalk — that the mechanistic model does not account for.

A Neural Differential Equation (NDE) is appended as an additive correction to the
mechanistic RHS.  After training on trajectory data the NDE recovers the hidden periodic
term.

```
dQ/dt = k_ind·I − k_rel·Q         ← mechanistic  (model.to_jax())
       + nn([t̄, Q̄, Ī])            ← unknown forcing  (Equinox MLP, to be learned)
```

## 1 · Mechanistic fluorescence model

The quenching state $Q$ builds up under illumination and relaxes spontaneously:

$$\frac{dQ}{dt} = k_\text{ind}\cdot I - k_\text{rel}\cdot Q$$

Fluorescence yield follows the Stern–Volmer relationship:

$$F(t) = \frac{1}{1 + Q(t)}$$

where $I$ is the photon flux density (PFD, μmol m⁻² s⁻¹) treated as a model parameter
that changes with each protocol phase.

In [ ]:
def linear_induction(k_ind: float, I: float) -> float:
    """Quenching induction proportional to light intensity."""
    return k_ind * I


def mass_action_relaxation(k_rel: float, Q: float) -> float:
    """First-order quenching relaxation."""
    return k_rel * Q


model = (
    Model()
    .add_variable("Q", 0.0)
    .add_parameters({"k_ind": 0.001, "k_rel": 0.1, "I": 100.0})
    .add_reaction(
        "induction",
        linear_induction,
        stoichiometry={"Q": 1},
        args=["k_ind", "I"],
    )
    .add_reaction(
        "relaxation",
        mass_action_relaxation,
        stoichiometry={"Q": -1},
        args=["k_rel", "Q"],
    )
)

print("Variables :", model.get_variable_names())
print("Parameters:", list(model.get_parameter_values()))

## 2 · Illumination protocol

Three phases of 100 s each, defined as an mxlpy `pd.DataFrame` protocol:

| Phase | Duration | PFD |
|-------|----------|-----|
| Low light   | 0 – 100 s   | 100 PFD  |
| High light  | 100 – 200 s | 2000 PFD |
| Medium light| 200 – 300 s | 500 PFD  |

Each row updates the parameter `I` for that segment.

In [ ]:
protocol = pd.DataFrame(
    {"I": [100.0, 2000.0, 500.0]},
    index=pd.to_timedelta([100, 200, 300], unit="s"),
)

T_TOTAL = 300.0

# Parse for segment-wise scipy integration (true data generation)
segments: list[tuple[float, float, float]] = []
t_cursor = 0.0
for td, row in protocol.iterrows():
    t_end = td.total_seconds()
    segments.append((t_cursor, t_end, float(row["I"])))
    t_cursor = t_end

I_MAX = float(protocol["I"].max())
t_edges = [0.0] + [t1 for _, t1, _ in segments]
I_vals = [I for _, _, I in segments]

fig, ax = plt.subplots(figsize=(9, 2.5))
ax.stairs(I_vals, t_edges, color="goldenrod", linewidth=2.5)
ax.set(xlabel="Time (s)", ylabel="PFD (μmol m⁻² s⁻¹)", title="Illumination protocol")
ax.set_ylim(0, 2400)
plt.tight_layout()

## 3 · Export the mechanistic model to JAX

`model.to_jax()` converts the mxlpy model to a differentiable RHS via its SymPy
symbolic representation.  The returned `JaxExport` is callable as `f(t, y, args)`,
compatible with `diffrax.ODETerm`, and supports `jax.jit`, `jax.grad`, `jax.vmap`.

In [ ]:
export = model.to_jax()

print("variable_names :", export.variable_names)
print("parameter_names:", export.parameter_names)

# Sanity check: exported RHS matches model() at t=0
y0 = export.get_y0()
args0 = export.get_args()
dydt_jax = export(0.0, y0, args0)  # JaxExport is callable
dydt_model = model(0.0, list(y0))
print(f"\nRHS at t=0, Q=0:  JAX={float(dydt_jax[0]):.5f}  model={dydt_model[0]:.5f}")

## 4 · Synthetic data — true model with hidden sinusoidal forcing

The *true* system includes a periodic term absent from the mechanistic model:

$$\frac{dQ}{dt} = k_\text{ind}\cdot I - k_\text{rel}\cdot Q
   + \underbrace{A\sin(\omega t)}_{\text{hidden forcing}}$$

We integrate this segment by segment — mirroring the protocol structure — using scipy.

In [ ]:
A_TRUE = 0.3
OMEGA_TRUE = 2 * np.pi / 20.0
K_IND = model.get_parameter_values()["k_ind"]
K_REL = model.get_parameter_values()["k_rel"]

t_data = np.linspace(0.0, T_TOTAL, 301)

t_parts: list[np.ndarray] = []
Q_parts: list[np.ndarray] = []
y_cur = [0.0]

for i, (t0_seg, t1_seg, I_val) in enumerate(segments):
    if i == 0:
        t_pts = t_data[(t_data >= t0_seg) & (t_data <= t1_seg)]
    else:
        t_pts = t_data[(t_data > t0_seg) & (t_data <= t1_seg)]

    def true_rhs(t: float, y: np.ndarray, I: float = I_val) -> list[float]:
        return [K_IND * I - K_REL * y[0] + A_TRUE * np.sin(OMEGA_TRUE * t)]

    sol = solve_ivp(
        true_rhs,
        [t0_seg, t1_seg],
        y_cur,
        t_eval=t_pts,
        method="RK45",
        rtol=1e-7,
        atol=1e-9,
    )
    y_cur = [float(sol.y[0, -1])]
    t_parts.append(sol.t)
    Q_parts.append(sol.y[0])

t_data = np.concatenate(t_parts)
Q_true = np.concatenate(Q_parts)
F_true = 1.0 / (1.0 + Q_true)

print(
    f"Data points: {len(t_data)}  |  Q range: [{Q_true.min():.2f}, {Q_true.max():.2f}]"
)

fig, (ax1, ax2, ax3) = plt.subplots(3, 1, figsize=(9, 7), sharex=True)
ax1.stairs(I_vals, t_edges, color="goldenrod", lw=2)
ax1.set(ylabel="PFD", title="Illumination protocol")
ax2.plot(t_data, Q_true, color="steelblue", lw=1.5)
ax2.set(ylabel="Q", title="Quenching state (true, with hidden forcing)")
ax3.plot(t_data, F_true, color="forestgreen", lw=1.5)
ax3.set(xlabel="Time (s)", ylabel="F", title="Fluorescence yield (true)")
plt.tight_layout()

## 5 · Mechanistic reference simulation

`Simulator.simulate_protocol_time_course` integrates the model over all protocol phases,
updating the `I` parameter between segments automatically.

In [ ]:
mech_sim = Simulator(model).simulate_protocol_time_course(protocol, t_data)
mech_df = pd.concat(mech_sim.variables)
Q_mech = mech_df["Q"].to_numpy()
F_mech = 1.0 / (1.0 + Q_mech)

fig, axes = plt.subplots(2, 1, figsize=(9, 5), sharex=True)
axes[0].plot(t_data, Q_true, "k-", lw=1.5, label="True (with hidden forcing)")
axes[0].plot(t_data, Q_mech, "b--", lw=1.5, label="Mechanistic only")
axes[0].set(ylabel="Q", title="Quenching — true vs. mechanistic")
axes[0].legend()
axes[1].plot(t_data, F_true, "k-", lw=1.5, label="True")
axes[1].plot(t_data, F_mech, "b--", lw=1.5, label="Mechanistic only")
axes[1].set(xlabel="Time (s)", ylabel="F", title="Fluorescence — true vs. mechanistic")
axes[1].legend()
plt.tight_layout()

## 6 · Building the Universal Differential Equation

We subclass `mxlpy.jax.UDE` (an `eqx.Module`) and override `__call__` to add the
neural correction.  The base class stores `ode` (the mechanistic export) as a static
field so JAX never differentiates through it — only the MLP weights receive gradients.

$$\frac{dQ}{dt} = \underbrace{k_\text{ind}\cdot I - k_\text{rel}\cdot Q}_{\text{mechanistic}}
+\; \underbrace{\text{MLP}(\bar{t},\, \bar{Q},\, \bar{I})}_{\text{neural correction}}$$

The light intensity $I$ is fixed per protocol segment and read directly from `args` —
no piecewise time function needed inside the ODE.

In [ ]:
Q_MAX = float(Q_true.max()) * 1.5
I_IDX = export.parameter_names.index("I")


class FluorescenceUDE(UDE):
    """Mechanistic fluorescence model + additive MLP correction."""

    q_max: float
    i_max: float
    t_total: float
    i_idx: int

    def __call__(self, t: float, y: jax.Array, args: jax.Array) -> jax.Array:
        mech = self.ode.rhs(t, y, args)
        i_norm = args[self.i_idx] / self.i_max
        nn_in = jnp.array([t / self.t_total, y[0] / self.q_max, i_norm])
        return mech + self.mlp(nn_in)


key = jax.random.PRNGKey(0)
ude = FluorescenceUDE(
    ode=export,
    mlp=eqx.nn.MLP(3, 1, width_size=32, depth=2, key=key, activation=jnp.tanh),
    q_max=Q_MAX,
    i_max=I_MAX,
    t_total=T_TOTAL,
    i_idx=I_IDX,
)

n_params = sum(p.size for p in jax.tree_util.tree_leaves(eqx.filter(ude, eqx.is_array)))
print(f"Trainable parameters: {n_params}")

## 7 · Training

`fit_jax.protocol_time_course` handles the full training loop:

- Parses the protocol into segments and builds per-segment parameter vectors
- Integrates the UDE segment by segment with Diffrax, chaining end-states
- Computes MSE against the target data and runs the Optax optimizer

All heavy lifting — segment setup, `eqx.filter_jit`, `filter_value_and_grad` — is done
inside the function.

In [ ]:
# Wrap training data as a DataFrame (index = time, columns = variable names)
data = pd.DataFrame({"Q": Q_true}, index=t_data)

fit_result = fit_jax.protocol_time_course(
    ude,
    protocol,
    data,
    minimizer=fit_jax.OptaxMinimizer(
        lr=3e-3,
        method=optax.adam,
        n_epochs=400,
    ),
)

trained_ude = fit_result.ude
print(f"Final loss: {fit_result.losses[-1]:.6f}")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3))
ax.semilogy(fit_result.losses, color="royalblue", lw=1.5)
ax.set(xlabel="Epoch", ylabel="MSE loss (log scale)", title="UDE training loss")
ax.grid(True, alpha=0.3)
plt.tight_layout()

## 8 · Results

### 8.1 Trajectory fit

In [ ]:
# Re-simulate with the trained UDE using the same segment-wise approach
from diffrax import Dopri5, ODETerm, PIDController, SaveAt, diffeqsolve


def simulate_trained(ude_model: FluorescenceUDE) -> np.ndarray:
    """Reproduce the training simulation for plotting."""
    y = export.get_y0()
    all_ys: list = []
    for i, (t0_seg, t1_seg, I_val) in enumerate(segments):
        t_pts = (
            t_data[(t_data >= t0_seg) & (t_data <= t1_seg)]
            if i == 0
            else t_data[(t_data > t0_seg) & (t_data <= t1_seg)]
        )
        params_seg = export.get_args(**{"I": I_val})
        sol = diffeqsolve(
            ODETerm(ude_model),
            Dopri5(),
            t0=t0_seg,
            t1=t1_seg,
            dt0=0.5,
            y0=y,
            saveat=SaveAt(ts=t_pts),
            stepsize_controller=PIDController(rtol=1e-3, atol=1e-5),
            args=params_seg,
            max_steps=4096,
        )
        y = sol.ys[-1]
        all_ys.append(sol.ys)
    return np.array(jnp.concatenate(all_ys, axis=0))[:, 0]


Q_ude = simulate_trained(trained_ude)
F_ude = 1.0 / (1.0 + Q_ude)

fig, axes = plt.subplots(2, 1, figsize=(10, 6), sharex=True)
axes[0].plot(t_data, Q_true, "k-", lw=2, label="True (hidden forcing)")
axes[0].plot(t_data, Q_ude, "r--", lw=1.8, label="UDE (trained)")
axes[0].plot(t_data, Q_mech, "b:", lw=1.5, label="Mechanistic only")
axes[0].set(ylabel="Quenching state Q")
axes[0].legend()
axes[1].plot(t_data, F_true, "k-", lw=2, label="True")
axes[1].plot(t_data, F_ude, "r--", lw=1.8, label="UDE (trained)")
axes[1].plot(t_data, F_mech, "b:", lw=1.5, label="Mechanistic only")
axes[1].set(xlabel="Time (s)", ylabel="Fluorescence F")
axes[1].legend()
plt.suptitle("UDE vs. mechanistic model vs. true trajectory", y=1.01)
plt.tight_layout()

### 8.2 What did the NDE learn?

We evaluate the trained NDE along the true $Q$ trajectory and compare its output
to the hidden sinusoidal forcing $A\sin(\omega t)$.

In [ ]:
# Build I array for each time point from protocol segments
I_arr = np.zeros(len(t_data))
for i, (t0_seg, t1_seg, I_val) in enumerate(segments):
    mask = (
        (t_data >= t0_seg) & (t_data <= t1_seg)
        if i == 0
        else (t_data > t0_seg) & (t_data <= t1_seg)
    )
    I_arr[mask] = I_val

# Vectorised NDE evaluation along the true trajectory
nn_inputs = jnp.array(
    np.stack([t_data / T_TOTAL, Q_true / Q_MAX, I_arr / I_MAX], axis=1)
)
nde_output = np.array(jax.vmap(trained_ude.mlp)(nn_inputs)[:, 0])
true_forcing = A_TRUE * np.sin(OMEGA_TRUE * t_data)

fig, ax = plt.subplots(figsize=(10, 3.5))
ax.plot(t_data, true_forcing, "k-", lw=2, label=f"True forcing: {A_TRUE}·sin(ωt)")
ax.plot(t_data, nde_output, "r--", lw=1.8, label="NDE output (learned)")
ax.axhline(0, color="gray", lw=0.8, ls=":")
ax.set(
    xlabel="Time (s)",
    ylabel="dQ/dt contribution",
    title="Recovered sinusoidal forcing vs. true hidden term",
)
ax.legend()
plt.tight_layout()

### 8.3 Residual analysis

In [ ]:
residual_ude = Q_ude - Q_true
residual_mech = Q_mech - Q_true

fig, ax = plt.subplots(figsize=(10, 3))
ax.plot(t_data, residual_mech, "b:", lw=1.5, alpha=0.8, label="Mechanistic only")
ax.plot(t_data, residual_ude, "r--", lw=1.5, alpha=0.8, label="UDE (trained)")
ax.axhline(0, color="k", lw=0.8)
ax.set(xlabel="Time (s)", ylabel="Q_pred − Q_true", title="Prediction residuals")
ax.legend()
plt.tight_layout()

print(f"RMSE mechanistic only : {np.sqrt(np.mean(residual_mech**2)):.4f}")
print(f"RMSE UDE (trained)    : {np.sqrt(np.mean(residual_ude**2)):.4f}")